# Move selection: the teacher policy ("constructive smurfing")

**Milestones:** rating-targeted play (~1700), fallback behaviour, keep games close (README
Teaching; design goals "keep games close" and "punish habits, not playstyles").

Given a position, the teacher:

1. **Checks the player model.** If confidence < `min_confidence`, or the current game is out of
   distribution (`ood_score > max_ood_score`), it **falls back** to solid, strength-limited play at
   the last estimated level (README: *"plays solid, strength-limited moves at its last estimated
   level and does not try to exploit a habit"*).
2. Otherwise gets **candidates** from the analyst (`multipv`) and keeps only those that give up at
   most `max_eval_drop_cp` versus the best move and never walk into a forced mate.
3. **Scores** each candidate: `w_close * closeness + w_target * weakness_target`, where closeness
   prefers evals inside `+-closeness_band_cp` and `weakness_target` (TODO) rewards moves that set
   the player a problem their profile says they get wrong (e.g. a position where the natural reply
   is a capture they tend to miss).

This class implements `MovePolicy.choose_move(board)` from `gambit_game/notebooks/game_manager.ipynb`,
so it plugs straight into the game manager.

**TODO checklist**
- [x] Fallback logic, candidate band filter, scoring skeleton, decision record.
- [ ] `weakness_target_score` heuristics (one per weakness in `PlayerModel.weaknesses()`).
- [ ] Weights `w_close` / `w_target`; maybe reveal more strength as the player improves (README).
- [ ] Open question: should Gambit ever play at full strength to avoid losing? (disclosure.ipynb)

In [ ]:
%run ./player_model.ipynb

In [ ]:
import math
from dataclasses import dataclass, field
from typing import Callable, Optional

import chess

## Decision record (logged with every robot move so experiments can be explained later)

In [ ]:
FALLBACK_LOW_CONFIDENCE = "fallback_low_confidence"
FALLBACK_OOD = "fallback_ood"
FALLBACK_NO_CANDIDATES = "fallback_no_candidates"
TEACHING = "teaching"


@dataclass
class TeacherDecision:
    move: chess.Move
    reason: str
    confidence: float
    ood_score: float
    candidates: list = field(default_factory=list)       # Candidates considered
    scores: dict = field(default_factory=dict)           # uci -> score

    @property
    def is_fallback(self) -> bool:
        return self.reason != TEACHING

## Candidate filtering and closeness

In [ ]:
def band_filter(cands: list[Candidate], max_drop: int = TEACH_CFG["max_eval_drop_cp"]) -> list[Candidate]:
    """Keep candidates within `max_drop` of the best and not losing to a forced mate."""
    if not cands:
        return []
    best = max(c.score_cp for c in cands)
    return [c for c in cands if best - c.score_cp <= max_drop and not (c.mate is not None and c.mate < 0)]


def closeness_score(score_cp: int, band: int = TEACH_CFG["closeness_band_cp"]) -> float:
    """1 inside the band around 0, decreasing linearly to 0 at 3x the band (robot POV)."""
    a = abs(score_cp)
    if a <= band:
        return 1.0
    return max(0.0, 1.0 - (a - band) / (2 * band))


def weakness_target_score(board: chess.Board, cand: Candidate, estimate: PlayerEstimate) -> float:
    """How well `cand` tests one of the player's weaknesses, in 0..1.

    TODO: for each Weakness (ordered by `low` bound, i.e. most confident first):
      * missed_forcing: after `cand`, does the player's best reply exist and is it a capture/check
        that is easy to overlook (e.g. a quiet-looking capture, a backward capture)?
      * blunder: does `cand` create a position where many natural moves lose material (count
        legal replies whose eval drops > blunder threshold, via a quick analyst call)?
      * phase weakness: steer toward the phase with the highest ACPL (trade into endgames, etc.).
    Punish habits, not playstyles: never score a move higher just because it is aggressive.
    """
    raise NotImplementedError("weakness_target_score")

## The policy

In [ ]:
class TeacherPolicy:
    def __init__(self, player_engine: EngineLike, analyst: EngineLike, model: PlayerModel,
                 cfg: dict = TEACH_CFG, multipv: int = ENGINE_CFG["multipv"],
                 target_fn: Callable = weakness_target_score, w_close: float = 1.0, w_target: float = 1.0,
                 min_live_moves: int = 10):
        self.player_engine, self.analyst, self.model = player_engine, analyst, model
        self.cfg, self.multipv, self.target_fn = cfg, multipv, target_fn
        self.w_close, self.w_target = w_close, w_target
        self.min_live_moves = min_live_moves
        self.live_analyses: list[MoveAnalysis] = []     # the human's moves in the current game
        self.decisions: list[TeacherDecision] = []

    def observe_human_move(self, analysis: MoveAnalysis) -> None:
        """Feed each analysed human move during the game (for the live OOD check)."""
        self.live_analyses.append(analysis)

    def live_ood_score(self) -> float:
        if len(self.live_analyses) < self.min_live_moves:
            return 0.0                                  # too few moves to judge this game yet
        return self.model.ood_score(self.live_analyses)

    def solid_move(self, board: chess.Board, estimate: Optional[PlayerEstimate]) -> chess.Move:
        """Strength-limited engine move at the last estimated level (or the default Elo)."""
        if hasattr(self.player_engine, "set_strength"):
            elo = int(estimate.elo) if estimate and estimate.n_games else ENGINE_CFG["default_elo"]
            self.player_engine.set_strength(elo=elo)
        return self.player_engine.play(board)

    def decide(self, board: chess.Board) -> TeacherDecision:
        est = self.model.estimate() if self.model.games else None
        conf = est.confidence if est else 0.0
        ood = self.live_ood_score()
        if conf < self.cfg["confidence"]["min_confidence"]:
            return self._record(TeacherDecision(self.solid_move(board, est), FALLBACK_LOW_CONFIDENCE, conf, ood))
        if ood > self.cfg["confidence"]["max_ood_score"]:
            return self._record(TeacherDecision(self.solid_move(board, est), FALLBACK_OOD, conf, ood))
        cands = band_filter(self.analyst.analyse(board, multipv=self.multipv), self.cfg["max_eval_drop_cp"])
        if not cands:
            return self._record(TeacherDecision(self.solid_move(board, est), FALLBACK_NO_CANDIDATES, conf, ood))
        scores = {}
        for c in cands:
            scores[c.move.uci()] = (self.w_close * closeness_score(c.score_cp, self.cfg["closeness_band_cp"])
                                    + self.w_target * self.target_fn(board, c, est))
        best = max(cands, key=lambda c: scores[c.move.uci()])
        return self._record(TeacherDecision(best.move, TEACHING, conf, ood, cands, scores))

    def _record(self, d: TeacherDecision) -> TeacherDecision:
        self.decisions.append(d)
        return d

    def choose_move(self, board: chess.Board) -> chess.Move:      # MovePolicy
        return self.decide(board).move

## Self-check

In [ ]:
c = [Candidate(chess.Move.from_uci("e2e4"), 300), Candidate(chess.Move.from_uci("d2d4"), 220),
     Candidate(chess.Move.from_uci("a2a3"), 100), Candidate(chess.Move.from_uci("h2h4"), 250, mate=-3)]
assert [x.move.uci() for x in band_filter(c, 120)] == ["e2e4", "d2d4"]
assert closeness_score(100) == 1.0 and closeness_score(-150) == 1.0 and closeness_score(450) == 0.0

# 1) no history -> low-confidence fallback
pol = TeacherPolicy(MaterialEngine(depth=1), MaterialEngine(depth=1), PlayerModel("new"))
d = pol.decide(chess.Board())
assert d.reason == FALLBACK_LOW_CONFIDENCE and d.move in chess.Board().legal_moves

# 2) confident model -> teaching path (stub target fn), prefers the closer move within the band.
#    Position: white can win a queen (+900) or keep it level; the teacher keeps it close only if
#    the drop is allowed, so with max_drop=120 it must still take the queen.
confident = PlayerModel("regular")
for i in range(8):
    confident.add_game_summary(_synthetic(f"g{i}", 55 + (i % 3) * 5))
pol = TeacherPolicy(MaterialEngine(depth=1), MaterialEngine(depth=1), confident, target_fn=lambda b, c, e: 0.0)
b = chess.Board("4k3/8/8/3q4/8/8/8/3QK3 w - - 0 1")
d = pol.decide(b)
assert d.reason == TEACHING and d.move.uci() == "d1d5", d
# with a wide allowed drop, closeness wins: it does not grab the queen and go +900
cfg_wide = {**TEACH_CFG, "max_eval_drop_cp": 2000}
pol_wide = TeacherPolicy(MaterialEngine(depth=1), MaterialEngine(depth=1), confident, cfg=cfg_wide,
                         target_fn=lambda b, c, e: 0.0, multipv=50)
d = pol_wide.decide(b)
assert d.reason == TEACHING and d.move.uci() != "d1d5" and abs(d.scores[d.move.uci()] - 1.0) < 1e-9

# 3) OOD during the game -> fallback
pol.live_analyses = [MoveAnalysis("live", i, "white", chess.STARTING_FEN, "e2e4", "d2d4", 0, -400, 400,
                                  "blunder", "middlegame", False, False) for i in range(12)]
assert pol.decide(b).reason == FALLBACK_OOD
print("move_selection self-check passed")